# Session 1: where does income catch up with cost?
**A. Real-world problem.** You want to run a lemonade stall. Before you sell a single cup
you must pay for the stall and the pitcher. After that, every cup costs a little to make and
earns a little more when it is sold. At the start you are behind. How many cups until you
are even? That number is the *break-even point*.

*General education only: the stall is made up, and this is not financial advice.*

**Plan.** (1) write the cost line, (2) write the income line, (3) find where they meet,
(4) draw them and read the graph.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

FIXED = 60.0      # dollars paid before the first cup: stall fee 40 + pitcher and sign 20
PRICE = 2.00      # dollars received for one cup
CUP_COST = 0.50   # dollars spent making one cup (lemons, sugar, cup)

## Step 1: the cost line
Two kinds of cost. The **fixed cost** is paid once, whether you sell 0 cups or 100.
The **variable cost** grows with every cup you make.

    total cost = fixed cost + cost per cup * cups

In [ ]:
def total_cost(cups, fixed=FIXED, cup_cost=CUP_COST):
    """Dollars spent after making `cups` cups."""
    return fixed + cup_cost * np.asarray(cups, dtype=float)

## Step 2: the income line and the profit
Income (also called revenue) is price times cups. Profit is income minus cost. While profit
is negative you are making a loss; once it is zero or more, you have caught up.

In [ ]:
def income(cups, price=PRICE):
    return price * np.asarray(cups, dtype=float)


def profit(cups, fixed=FIXED, price=PRICE, cup_cost=CUP_COST):
    return income(cups, price) - total_cost(cups, fixed, cup_cost)

## Step 3: where the lines meet
Each cup earns `price - cup cost` dollars towards paying off the fixed cost. So the number
of cups needed is the fixed cost divided by what one cup earns:

    break-even cups = fixed cost / (price - cup cost)

If a cup earns nothing or less (price <= cup cost), the lines never meet. We return `None`.

In [ ]:
def break_even_cups(fixed=FIXED, price=PRICE, cup_cost=CUP_COST):
    earn = price - cup_cost
    if earn <= 0:
        return None
    return fixed / earn


def first_profitable_cup(fixed=FIXED, price=PRICE, cup_cost=CUP_COST, limit=1000):
    """Count whole cups one by one until income >= cost. Returns None if never within `limit`."""
    cups = np.arange(0, limit + 1)
    ok = np.nonzero(profit(cups, fixed, price, cup_cost) >= -1e-9)[0]
    return int(cups[ok[0]]) if len(ok) else None

## Step 4: see it
Left: the two lines, with the meeting point marked. Right: profit after each batch of cups.
Below the zero line you are in loss; above it, in profit.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for c in (0, 10, 25, 39, 40, 41, 60):
        print(f"cups {c:3d}: cost {float(total_cost(c)):7.2f}  income {float(income(c)):7.2f}  profit {float(profit(c)):7.2f}")
    q = break_even_cups()
    print("break-even cups:", q, " counted one by one:", first_profitable_cup())
    print("price 0.50 (same as cup cost) ->", break_even_cups(price=0.50))

    cups = np.arange(0, 101)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(cups, total_cost(cups), color="#d9534f", label="total cost")
    ax.plot(cups, income(cups), color="#2a9d8f", label="income")
    ax.plot([q], [float(income(q))], "ko")
    ax.annotate(f"break-even: {q:.0f} cups, ${float(income(q)):.0f}", (q, float(income(q))), (47, 40))
    ax.set_xlabel("cups sold (cups)")
    ax.set_ylabel("dollars ($)")
    ax.set_title("Income catches up with cost")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "stall_lines.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    pr = profit(cups)
    ax2.bar(cups, pr, color=np.where(pr >= 0, "#2a9d8f", "#d9534f"), width=0.9)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_xlabel("cups sold (cups)")
    ax2.set_ylabel("profit ($)")
    ax2.set_title("Profit starts at -60 dollars and gains 1.50 dollars per cup")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "stall_profit.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `PRICE` in `lesson.py` to 3.00 dollars and predict the break-even before you run it. Then check with `break_even_cups`.
2. **Stretch.** Write `profit_after(cups)` for a stall with a second, extra fixed cost of 20 dollars that you pay after the 50th cup, and find where profit first becomes zero.
3. **Explain.** Explain, in your own words and with a sketch, why the income line must be steeper than the cost line for a break-even to exist.

## Assessment
- **A. Conceptual.** A friend says: the stall costs 60 dollars and each cup sells for 2 dollars, so break-even is 60 / 2 = 30 cups. What did they forget?
- **B. Numerical.** A different stall has fixed cost 45 dollars, price 1.50 dollars and a cup cost of 0.60 dollars. Find the break-even cups and check that income equals cost there.
- **C. Graph and simulation interpretation.** On the graph of the first stall, read the cost, the income and the profit at 25 cups. Is the stall ahead or behind?
- **D. Design and optimisation challenge.** Design: the stall will only be open for a short day. Change either the price or the fixed cost so that break-even is at 30 cups or fewer. Which would you pick and what is the risk?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("before any cup is sold the cost is the fixed $60", abs(float(L.total_cost(0)) - 60.0) < 1e-9))
    q = L.break_even_cups()
    checks.append(("worked example: break-even is 60 / (2.00 - 0.50) = 40 cups", abs(q - 40.0) < 1e-9))
    checks.append(("worked example: at 40 cups income = cost = $80",
                   abs(float(L.income(40)) - 80.0) < 1e-9 and abs(float(L.total_cost(40)) - 80.0) < 1e-9))
    checks.append(("counting cups one by one also gives 40", L.first_profitable_cup() == 40))
    checks.append(("39 cups: still $1.50 short", abs(float(L.profit(39)) + 1.5) < 1e-9))
    checks.append(("41 cups: $1.50 ahead", abs(float(L.profit(41)) - 1.5) < 1e-9))
    checks.append(("25 cups: income $50, cost $72.50, loss $22.50",
                   abs(float(L.income(25)) - 50.0) < 1e-9 and abs(float(L.total_cost(25)) - 72.5) < 1e-9
                   and abs(float(L.profit(25)) + 22.5) < 1e-9))
    checks.append(("each extra cup adds $1.50 to the profit", abs(float(L.profit(11) - L.profit(10)) - 1.5) < 1e-9))
    checks.append(("hand calculation F=45, price 1.50, cup cost 0.60 gives 50 cups",
                   abs(L.break_even_cups(45.0, 1.5, 0.6) - 50.0) < 1e-9
                   and abs(float(L.income(50, 1.5)) - 75.0) < 1e-9))
    checks.append(("price $2.50 moves break-even to 30 cups", abs(L.break_even_cups(price=2.5) - 30.0) < 1e-9))
    checks.append(("fixed cost $45 at price $2.00 also gives 30 cups", abs(L.break_even_cups(fixed=45.0) - 30.0) < 1e-9))
    checks.append(("price $1.00 needs 120 cups", abs(L.break_even_cups(price=1.0) - 120.0) < 1e-9))
    checks.append(("limit: price equal to the cup cost has no break-even", L.break_even_cups(price=0.5) is None))
    checks.append(("limit: price below the cup cost has no break-even", L.break_even_cups(price=0.3) is None
                   and L.first_profitable_cup(price=0.3) is None))
    checks.append(("limit: no fixed cost means break-even at 0 cups", L.break_even_cups(fixed=0.0) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
